# First Multi-GPU Attempt

At this point, we are able to target our first multi-GPU implementation.
Our approach is now straightforward:

Instead of running **multiple streams** on **one device**, use **multiple devices** with **one stream each**.

### 1. Determine the Number of GPUs

A robust and portable implementation works for an arbitrary number of GPUs exposed to it.
The first step is querying this number from CUDA.

```cpp
int numDevices = 0;
checkCudaError(cudaGetDeviceCount(&numDevices));
```


Once we have this information, we can use it to determine the number of patches.
In the easiest case, this will be a one-to-one mapping.

```cpp
int numPatches = numDevices; // one patch per device
```

By setting the number of patches to the number of GPUs, we can reuse our existing patch construction code.

Note that `cudaGetDeviceCount` reports the number of devices **visible** to the application.
The `CUDA_VISIBLE_DEVICES` environment variable can be used to limit the set of GPUs exposed.

### 2. Synchronization with Multiple GPUs

In contrast to streams, device selection doesn't work on a per-operation basis in CUDA.
Instead, CUDA tracks a single **selected** or **active GPU**.
All API calls, memory operations, and kernel launches apply exclusively to this currently active device.

The active device can be queried via `cudaGetDevice` and set through `cudaSetDevice`.

```cpp
int deviceId = -1;
checkCudaError(cudaGetDevice(&deviceId));   // pointer argument
std::cout << "Currently active device: " << deviceId << std::endl;
```


This also means, that full synchronization requires additional effort.
Instead of synchronizing with a single GPU (and all its streams), we now need to ensure that **all** patches on **all** GPUs have finished.

```cpp
for (int deviceIdx = 0; deviceIdx < numDevices; ++deviceIdx) {
    checkCudaError(cudaSetDevice(deviceIdx));
    checkCudaError(cudaDeviceSynchronize());
}
```


### 3. Multi-GPU Kernel Launches

The final step is adapting our computation once again.
Instead of looping over streams and pushing one patch into each, we now loop over the devices and launch one kernel each.
Note the added `cudaSetDevice` as first statement of the device loop.

```cpp
for (int deviceIdx = 0; deviceIdx < numDevices; ++deviceIdx) {
    checkCudaError(cudaSetDevice(deviceIdx));

    const auto &patch = patches[deviceIdx];

    stencil2D<<<patch.gridSize, patch.blockSize>>>(...);
}

for (int deviceIdx = 0; deviceIdx < numDevices; ++deviceIdx) {
    checkCudaError(cudaSetDevice(deviceIdx));
    checkCudaError(cudaDeviceSynchronize());
}

std::swap(u, uNew);
```


## Exercise

Choose one of the difficulty levels below to tailor the exercise to your preferences.
Each level provides a different starting point implementation to be copied into [stencil-2d.cu](../src/06-mgpu/stencil-2d.cu).
Use it for your implementation and follow the steps outlined above.

Below the difficulty level descriptions, there are cells for compiling, executing and profiling the your solution.

### Level Hard

Create and empty file [stencil-2d.cu](../src/06-mgpu/stencil-2d.cu) and copy one of the previous code versions into it (your work or a solution).

In [ ]:
!touch ../src/06-mgpu/stencil-2d.cu

### Level Medium

[stencil-2d-medium.cu](../src/06-mgpu/stencil-2d-medium.cu) contains a partial solution with TODOs ranging from straight-forward to complex.
Copy the provided code into the working file [stencil-2d.cu](../src/06-mgpu/stencil-2d.cu) with the cell below, then finish the implementation provided there.

In [ ]:
!cp ../src/06-mgpu/stencil-2d-medium.cu ../src/06-mgpu/stencil-2d.cu

### Level Easier

[stencil-2d-easier.cu](../src/06-mgpu/stencil-2d-easier.cu) contains a partial solution with TODOs.
The solution is further progressed than the level medium version, and the complexity of the TODOs is limited.
Copy the provided code into the working file [stencil-2d.cu](../src/06-mgpu/stencil-2d.cu) with the cell below, then finish the implementation provided there.

In [ ]:
!cp ../src/06-mgpu/stencil-2d-easier.cu ../src/06-mgpu/stencil-2d.cu

### Possible Solution

[stencil-2d-solution.cu](../src/06-mgpu/stencil-2d-solution.cu) contains a possible solution for this exercise.
Copy it into the working file [stencil-2d.cu](../src/06-mgpu/stencil-2d.cu) with the cell below.

In [ ]:
!cp ../src/06-mgpu/stencil-2d-solution.cu ../src/06-mgpu/stencil-2d.cu

### Compilation, Execution and Profiling

The new code version is available in [06-mgpu/stencil-2d.cu](../src/06-mgpu/stencil-2d.cu) (after creating it with one of the commands above).
It can be compiled and executed with the following cells.

In [ ]:
!nvcc -O3 -gencode arch=compute_80,code=sm_80 -gencode arch=compute_86,code=sm_86 -o ../build/06-mgpu ../src/06-mgpu/stencil-2d.cu

The next cell produces output for a small grid.
Visualize the output using the [visualize](./99-visualize.ipynb) notebook after executing the application.

In [ ]:
!../build/06-mgpu 256 64 2 2000 100

The next cell produces no output and runs a larger grid.
Use it for performance evaluation.

In [ ]:
!../build/06-mgpu $((32 * 1024)) 256 2 16 0

Instead of running on the local **single A40** GPU, we can also submit a batch job running on **multiple A100** GPUs.
Feel free to tune the number of GPUs (both in the `--gres=gpu:a100:NGPU` and in the `mpirun -n NGPU`) to anything between one and eight GPUs.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:2 \
    --time 00:05:00 --wait \
    --output=../output/06-mgpu.out --error=../output/06-mgpu.err \
    --wrap="../build/06-mgpu $((32 * 1024)) 256 2 8 0"

cat ../output/06-mgpu.out

The next cell performs profiling with Nsight Systems by submitting a batch job.
Feel free to tune the number of GPUs in the `--gres=gpu:a100:NGPU` to anything between one and eight GPUs.

The profile is then available at [profiles/06-mgpu.nsys-rep](../profiles/06-mgpu.nsys-rep) and can be downloaded by **shift + right-clicking** the link, by clicking the link with the **middle mouse button**, or using the JupyterHub file tree.

After downloading it, open it up locally to visualize the run-time behavior of your application.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:2 \
    --time 00:05:00 --wait \
    --output=../output/06-mgpu-nsys.out --error=../output/06-mgpu-nsys.err \
    --wrap="nsys profile --stats=true --force-overwrite=true \
        -o ../profiles/06-mgpu \
        ../build/06-mgpu $((32 * 1024)) 256 2 8 0"

cat ../output/06-mgpu-nsys.out

## Next Step

The next step is optimizing the implementation with per-patch device allocations [07](./07-halos.ipynb).